In [2]:
import numpy as np

In [3]:
# least squares

def least_squares(x, y):
    n = len(x)
    m = (n*sum(x*y) - sum(x)*sum(y)) / (n*sum(x*x) - sum(x)**2)
    c = (sum(y) - m*sum(x)) / n

    return m, c

In [ ]:
# polynomial fitting 



In [4]:
# cubic spine


def setup_tridiagonal_matrix(x, y):
    n = len(x)
    h = np.diff(x)  # Step sizes between data points
    # Initialize the tridiagonal matrix coefficients
    A = np.zeros(n - 2)  # Subdiagonal
    B = np.zeros(n - 2)  # Main diagonal
    C = np.zeros(n - 2)  # Superdiagonal
    D = np.zeros(n - 2)  # Right-hand side vector
    # Populate the tridiagonal matrix coefficients
    for i in range(1, n - 1):
        A[i - 1] = h[i-1]               # Subdiagonal
        B[i - 1] = 2*(h[i-1]+h[i])      # Main diagonal
        C[i - 1] = h[i]                 # Superdiagonal
        D[i - 1] = 6*((y[i+1]-y[i])/h[i] - (y[i]-y[i-1])/h[i-1])    # Right-hand side vector
    return A, B, C, D


def thomas_algorithm(a, b, c, d):
    """
    Solve the tridiagonal system using the Thomas algorithm.
    a: sub-diagonal elements (length n-1)
    b: main diagonal elements (length n)
    c: super-diagonal elements (length n-1)
    d: right-hand side (length n)
    Returns: solution vector x of length n
    """
    n = len(b)
    # Forward elimination
    for i in range(1, n):
        w = a[i-1] / b[i-1]
        b[i] = b[i] - w * c[i-1]
        d[i] = d[i] - w * d[i-1]
    # Back substitution
    x = np.zeros(n)
    x[-1] = d[-1] / b[-1]
    for i in range(n-2, -1, -1):
        x[i] = (d[i] - c[i] * x[i+1]) / b[i]
    return x


def cubic_spline(X, Y):
    "natural cubic spline solver"
    H = np.diff(X)

    # Set up the tridiagonal matrix for the cubic spline
    A, B, C, D = setup_tridiagonal_matrix(X, Y)

    M = [0, *thomas_algorithm(A, B, C, D), 0]

    # construct splines
    curves = [
        lambda x, M=M, H=H, X=X, Y=Y, i=i: (
            M[i]/6/H[i] * (X[i+1] - x)**3 + 
            M[i+1]/6/H[i] * (x - X[i])**3 +
            (Y[i]/H[i] - M[i]*H[i]/6) * (X[i+1] - x) +
            (Y[i+1]/H[i] - M[i+1]*H[i]/6) * (x - X[i])
        )
            for i in range(len(H))
    ]
    
    # solve cubic spline
    x_f = np.arange(X[0], X[-1], .01)
    y_f = np.zeros(len(x_f))

    for i, x_i in enumerate(x_f):
        c_i = np.argmax(X>x_i) - 1
        y_f[i] = curves[c_i](x_i)
    
    return x_f, y_f
